# Goal-conditioned CommNet

> CommNet conditioned on each agent's goal (a pose and/or an image), trained from datasets with hindsight goals, sharing latents (quantized with comm-core), raw observations and poses.

In [ ]:
#| default_exp wm.commnet.gc

In [ ]:
#| hide
from nbdev.showdoc import *

`GCCommNet` is `CommNet` (see `stable_marl.wm.commnet.core`) with a goal per agent, in the way of
the goal-conditioned baselines (GCRL, OmniVLA): it learns to reach any goal from data collected by
any policy. Each agent encodes its observation and its goal, `h_i^0 = tanh(W_s s_i + W_g g_i)`, with
the goal given as

* `pose`: the goal cell and direction in the agent's frame (`goal_pose`: forward, left, in units of
  the grid, and the turn);
* `image`: the agent's view at the goal (FindGoal's `goal`, or a view it had later in the episode),
  average-pooled to one color per cell of the view (`goal_image_cells`, e.g. 7 x 7 x 3 numbers);

or both (a random non-empty subset per agent and sample in training: modality dropout).

**What agents share.** Messages are bundles, as `CommunicationWrapper` sends them: one message per
agent and round holding everything it shares (`share`, any subset of):

* `latent`: its hidden state `h_k^j`, every round (CommNet's message), quantized when sent with
  comm-core's `UniformQuantizer` (`latent_bits` per number over the [-1, 1] of tanh; the gradient
  passes straight through the rounding), or float32 (`latent_bits=None`);
* `observation`: its raw observation (the arrays of `obs_keys`), in the first round's message;
  the receiver encodes it with its own encoder;
* `pose`: its proprioception (`position`, `direction`), in the first round's message; the
  receiver encodes the sender's pose relative to its own.

Goals are not shared as such: they reach the others only through latents, if the model learns to
put them there. Each kind is averaged over the senders heard (CommNet's mean) and enters the next
step through its own matrix (`C^j`, `C_obs`, `C_pose`). As in CommNet, `c^0 = 0`, so the first
messages arrive in step 1 and sharing needs `comm_steps >= 2`. The size of each round's bundle
(`message_bits`) is what comm-core networks (`NetworkLinks`) transmit, and a lost bundle loses
everything in it.

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from fastcore.utils import patch

from stable_marl.envs.multigrid.core.constants import Direction
from stable_marl.wm.commnet.core import CommNet, comm_mask, commnet_features, mean_of_others
from stable_marl.wm.omnivla.core import IGNORE_INDEX, goal_pose, shortest_path_actions, walkable_cells

### The network

In [ ]:
#| export
SHARE = ('latent', 'observation', 'pose')
GOAL_MODALITIES = ('pose', 'image')
_FORWARD = torch.tensor([Direction(d).to_vec() for d in range(4)], dtype=torch.float32)


def relative_poses(position: torch.Tensor, direction: torch.Tensor, grid_size: int) -> torch.Tensor:
    "`[forward, left, cos, sin]` (B, N, N, 4) of agent `k`'s pose in agent `i`'s frame (`[:, i, k]`), offsets in units of the grid."
    forward = _FORWARD.to(position.device)[direction.long()]                     # (B, N, 2)
    left = torch.stack([forward[..., 1], -forward[..., 0]], -1)
    offset = (position.unsqueeze(1) - position.unsqueeze(2)).float() / grid_size  # [:, i, k]: k - i
    yaw = -(direction.unsqueeze(1) - direction.unsqueeze(2)).float() * (torch.pi / 2)
    return torch.stack([(offset * forward.unsqueeze(2)).sum(-1), (offset * left.unsqueeze(2)).sum(-1),
                        torch.cos(yaw), torch.sin(yaw)], -1)


def _check(values: Sequence[str], allowed: tuple, what: str, empty: bool = True) -> tuple:
    unknown = set(values) - set(allowed)
    if unknown or (not empty and not values):
        raise ValueError(f"{what} must be a {'' if empty else 'non-empty '}subset of {allowed}, got {list(values)}")
    return tuple(v for v in allowed if v in values)


class GCCommNet(CommNet):
    "Goal-conditioned CommNet: every agent gets a goal (pose and/or image) and shares latents, raw observations and/or poses."
    def __init__(
        self,
        obs_dim: int, # Features per agent (see `commnet_features`, without the goal)
        goal_image_cells: int = 7, # Side of the agent's view in cells: goal images are pooled to one color per cell
        num_actions: int = 4,
        hidden_dim: int = 128,
        comm_steps: int = 2, # K (shared things arrive from step 1: K >= 2 to communicate)
        skip: bool = True,
        share_weights: bool = False,
        obs_keys: Sequence[str] = ('image', 'direction'),
        grid_size: int = 15,
        comm_range: int | None = None,
        drop_prob: float = 0.0,
        share: Sequence[str] = ('latent',), # What the agents send: latent, observation, pose (empty: nothing)
        latent_bits: int | None = 8, # Bits per latent number when sent (comm-core's UniformQuantizer); None: float32
        bits_per_element: int = 32, # Bits per number of the raw observations and poses (as `CommunicationWrapper`: float32)
        goal_modalities: Sequence[str] = GOAL_MODALITIES, # Goals the model can be given (each sample: a non-empty subset)
    ):
        super().__init__(obs_dim, num_actions, hidden_dim, comm_steps, skip, share_weights, obs_keys, goal=False,
                         grid_size=grid_size, comm_range=comm_range, drop_prob=drop_prob)
        self.share = _check(share, SHARE, 'share')
        self.goal_modalities = _check(goal_modalities, GOAL_MODALITIES, 'goal_modalities', empty=False)
        self.latent_bits, self.bits_per_element = latent_bits, bits_per_element
        self.config.pop('goal')                     # the goal is an input of its own here
        self.goal_image_cells = goal_image_cells
        self.config.update(goal_image_cells=goal_image_cells, share=list(self.share), latent_bits=latent_bits,
                           bits_per_element=bits_per_element, goal_modalities=list(self.goal_modalities))
        self.goal_encoder = nn.Linear(4 + 3 * goal_image_cells ** 2 + 2, hidden_dim)   # pose, image, which are given
        if 'observation' in self.share:
            self.C_obs = nn.Linear(hidden_dim, hidden_dim, bias=False)
        if 'pose' in self.share:
            self.pose_encoder = nn.Linear(4, hidden_dim)
            self.C_pose = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self._quantizer = None
        self.eval_modalities: Sequence[str] | None = None   # acting: the goals given (None: `goal_modalities`)

    def quantize(self, h: torch.Tensor) -> torch.Tensor:
        "`h` as received after `UniformQuantizer` (`latent_bits` per number over [-1, 1]), with a straight-through gradient."
        if self.latent_bits is None:
            return h
        if self._quantizer is None:
            from comm_core import UniformQuantizer
            self._quantizer = UniformQuantizer(self.latent_bits, -1.0, 1.0)
        q = self._quantizer.decode(self._quantizer.encode(h.detach())).to(h.dtype)
        return h + (q - h).detach()

    def forward(
        self,
        obs: torch.Tensor, # (B, N, obs_dim)
        position: torch.Tensor, # (B, N, 2) cells
        direction: torch.Tensor, # (B, N)
        goal_pose: torch.Tensor, # (B, N, 4) (see `goal_pose`, in cells)
        goal_image: torch.Tensor, # (B, N, H, W, 3) uint8
        goal_mask: torch.Tensor, # (B, N, 2) bool: whether the pose and the image are given
        mask: torch.Tensor | None = None, # (B, N, N) or per round (B, K - 1, N, N): who hears whom; None: everyone
    ) -> torch.Tensor:
        "Action logits (B, N, num_actions)."
        goal_mask = goal_mask.bool()
        pose = goal_pose.float() * torch.tensor([1 / self.grid_size, 1 / self.grid_size, 1, 1], device=obs.device)
        B, N = goal_image.shape[:2]
        cells = self.goal_image_cells
        image = goal_image.reshape(B * N, *goal_image.shape[2:]).permute(0, 3, 1, 2).float() / 255
        image = F.adaptive_avg_pool2d(image, (cells, cells)).reshape(B, N, -1)   # one color per cell
        goal = torch.cat([pose * goal_mask[..., :1], image * goal_mask[..., 1:], goal_mask.float()], -1)
        h0 = torch.tanh(self.encoder(obs) + self.goal_encoder(goal))
        h = h0
        for j in range(self.comm_steps):
            m = 0 if self.share_weights else j
            out = self.H[m](h)
            if j > 0:
                heard = mask[:, j - 1] if mask is not None and mask.dim() == 4 else mask
                if 'latent' in self.share:
                    out = out + self.C[0 if self.share_weights else j - 1](mean_of_others(self.quantize(h), heard))
                if j == 1 and 'observation' in self.share:   # raw observations, encoded by the receiver
                    out = out + self.C_obs(mean_of_others(torch.tanh(self.encoder(obs)), heard))
                if j == 1 and 'pose' in self.share:          # poses, relative to the receiver's
                    poses = torch.tanh(self.pose_encoder(relative_poses(position, direction, self.grid_size)))
                    out = out + self.C_pose(_mean_of_others_pairs(poses, heard))
            if self.S is not None:
                out = out + self.S[m](h0)
            h = torch.tanh(out)
        return self.policy_head(h)


def _mean_of_others_pairs(x: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:
    "Mean over `k != i` of `x[:, i, k]` (B, N, N, D) for the `k` agent `i` hears, 0 if none."
    B, N = x.shape[:2]
    hears = torch.ones(B, N, N, dtype=torch.bool, device=x.device) if mask is None else mask.bool()
    hears = (hears & ~torch.eye(N, dtype=torch.bool, device=x.device)).unsqueeze(-1).to(x.dtype)
    return (x * hears).sum(2) / hears.sum(2).clamp(min=1)

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
pos = torch.tensor([[[1, 1], [3, 2]]]); dirs = torch.tensor([[0, 3]])
rel = relative_poses(pos, dirs, grid_size=1)
test_close(rel[0, 0, 1], torch.from_numpy(goal_pose(np.array([1, 1]), 0, np.array([3, 2]), 3)))   # as `goal_pose`
test_close(rel[0, 1, 0], torch.from_numpy(goal_pose(np.array([3, 2]), 3, np.array([1, 1]), 0)), eps=1e-6)

torch.manual_seed(0)
B, N, D, G = 2, 3, 6, 2              # goal images of 4 x 4 pixels, pooled to 2 x 2 cells
args = dict(position=torch.randint(0, 5, (B, N, 2)), direction=torch.randint(0, 4, (B, N)), goal_pose=torch.randn(B, N, 4),
            goal_image=torch.randint(0, 256, (B, N, 4, 4, 3), dtype=torch.uint8), goal_mask=torch.ones(B, N, 2, dtype=torch.bool))
obs = torch.randn(B, N, D)
other = obs.clone(); other[:, 1] += 1
moved = {**args, 'position': args['position'].clone()}; moved['position'][:, 1] += 1
for share, by_obs, by_pose in (((), False, False), (('latent',), True, False), (('observation',), True, False),
                               (('pose',), False, True)):
    net = GCCommNet(D, goal_image_cells=G, hidden_dim=8, share=share)
    out = net(obs, **args)
    test_eq(not torch.allclose(out[:, 0], net(other, **args)[:, 0]), by_obs)        # agent 1's observation reaches agent 0
    test_eq(not torch.allclose(out[:, 0], net(obs, **moved)[:, 0]), by_pose)        # agent 1's pose reaches agent 0
net = GCCommNet(D, goal_image_cells=G, hidden_dim=8, share=SHARE, comm_steps=1)
test_close(net(obs, **args)[:, 0], net(other, **moved)[:, 0])                      # K = 1: nothing arrives
test_fail(lambda: GCCommNet(D, goal_image_cells=G, share=['goal']))

# the goal modalities: a masked one does not change the output
net = GCCommNet(D, goal_image_cells=G, hidden_dim=8)
pose_only = {**args, 'goal_mask': torch.tensor([True, False]).expand(B, N, 2)}
other_image = {**pose_only, 'goal_image': torch.randint(0, 256, (B, N, 4, 4, 3), dtype=torch.uint8)}
test_close(net(obs, **pose_only), net(obs, **other_image))

# quantized latents: what is sent takes 2**bits values, and the gradient passes through
net = GCCommNet(D, goal_image_cells=G, hidden_dim=8, latent_bits=2)
h = torch.tanh(torch.randn(4, 8, requires_grad=True))
q = net.quantize(h)
test_eq(len(torch.unique(q.detach())) <= 4, True)
q.sum().backward()

### Acting

`message_bits` is what each agent sends per round: the first round's bundle has the latent (if
shared) and the raw observation and pose (if shared), the later ones the latent only. With
comm-core networks (`links`), these are the sizes they transmit.

In [ ]:
#| export
@patch
def message_bits(self: GCCommNet, columns: dict[str, np.ndarray]) -> list[int]:
    "Bits per agent of each round's message (K - 1 rounds), for columns of one step (B, N, ...)."
    latent = self.config['hidden_dim'] * (self.latent_bits or 32) if 'latent' in self.share else 0
    first = latent
    if 'observation' in self.share:
        first += self.bits_per_element * sum(int(np.prod(np.asarray(columns[k]).shape[2:])) for k in self.obs_keys)
    if 'pose' in self.share:
        first += self.bits_per_element * 3                      # position (2) and direction
    return [first if r == 0 else latent for r in range(self.comm_steps - 1)]


@patch
def _info_keys(self: GCCommNet) -> list[str]:
    return list(dict.fromkeys([*self.obs_keys, 'terminated', 'position', 'direction', 'goal_position', 'goal']))


@patch
def observe(self: GCCommNet, columns: dict[str, np.ndarray], modalities: Sequence[str] | None = None) -> dict[str, torch.Tensor]:
    """
    The inputs of `forward` (and `mask`) for columns of one step (B, N, ...), with each agent's goal
    from `goal_position` (looking `right`, as FindGoal's goal image) and `goal`, given as `modalities`
    (default: `goal_modalities`).
    """
    modalities = _check(modalities or self.goal_modalities, GOAL_MODALITIES, 'modalities', empty=False)
    position, direction = np.asarray(columns['position']), np.asarray(columns['direction'])
    alive = ~np.asarray(columns['terminated'], bool)
    B, N = alive.shape
    inputs = {
        'obs': torch.from_numpy(commnet_features(columns, self.obs_keys, self.grid_size, goal=False)),
        'position': torch.from_numpy(position.astype(np.int64)), 'direction': torch.from_numpy(direction.astype(np.int64)),
        'goal_pose': torch.from_numpy(goal_pose(position, direction, np.asarray(columns['goal_position']), 0)),
        'goal_image': torch.as_tensor(np.asarray(columns['goal'])),
        'goal_mask': torch.tensor([m in modalities for m in GOAL_MODALITIES]).expand(B, N, 2),
    }
    mask = comm_mask(torch.from_numpy(alive), inputs['position'] if self.comm_range is not None else None,
                     self.comm_range, self.drop_prob)
    if self.links is not None and self.comm_steps > 1:
        mask = mask.unsqueeze(1) & self.links.masks(position, alive, self.comm_steps - 1, self.config['hidden_dim'],
                                                    message_bits=self.message_bits(columns))
    inputs['mask'] = mask
    return inputs


@patch
@torch.no_grad()
def get_action(self: GCCommNet, info: dict) -> torch.Tensor:
    "Actions (num_envs, num_agents) for the info dict of a pool of envs; goals from `goal_position` and `goal` (`eval_modalities`)."
    columns = {k: torch.as_tensor(info[k])[:, -1].cpu().numpy() for k in self._info_keys()}
    if self.links is not None:
        flush = info.get('_needs_flush')
        envs = range(len(columns['terminated'])) if flush is None else np.flatnonzero(torch.as_tensor(flush).cpu().numpy())
        seeds = torch.as_tensor(info['seed'])[:, -1].cpu().numpy() if 'seed' in info else None
        self.links.reset(envs, None if seeds is None else [int(seeds[e]) for e in envs])
    device = next(self.parameters()).device
    inputs = {k: v.to(device) for k, v in self.observe(columns, self.eval_modalities).items()}
    logits = self(**inputs)
    if self.temperature > 0:
        return torch.distributions.Categorical(logits=logits / self.temperature).sample()
    return logits.argmax(-1)

### Training data

`GCCommNetDataset` makes one sample per step of every episode (all the agents together), each
agent with a goal: its episode's goal, or with probability `hindsight_prob` a state it reached
later (its cell, direction and view then), at most `max_goal_steps` steps later. With `hindsight`
`'joint'` the agents' goals are taken at one common future step (the team's configuration then),
with `'agent'` each agent's at its own. The labels are the shortest-path expert's actions towards
each agent's goal (recomputed from the recorded grid, see `shortest_path_actions`), ignored for
agents that finished or stand on their goal. Each agent is given a random non-empty subset of the
model's `goal_modalities`.

In [ ]:
#| export
HINDSIGHT = ('joint', 'agent')


class GCCommNetDataset(torch.utils.data.Dataset):
    "Joint samples of a MultiGrid dataset with a goal per agent (the episode's or hindsight), labelled by the expert."
    def __init__(
        self,
        dataset, # Episodes with the model's `obs_keys`, `position`, `direction`, `pixels`, `goal`, `goal_position`, `terminated`, `state`
        model: GCCommNet,
        hindsight_prob: float = 0.5, # Probability of hindsight goals instead of the episode's goal
        hindsight: str = 'joint', # 'joint': all the agents' goals at one future step; 'agent': each at its own
        max_goal_steps: int = 20, # Farthest future step used as a goal
    ):
        if hindsight not in HINDSIGHT:
            raise ValueError(f"hindsight must be one of {HINDSIGHT}, got {hindsight!r}")
        self.hindsight_prob, self.hindsight, self.max_goal_steps = hindsight_prob, hindsight, max_goal_steps
        self.subsets = [m for m in ((True, False), (False, True), (True, True))
                        if all(GOAL_MODALITIES[i] in model.goal_modalities for i in range(2) if m[i])]
        keys = list(dict.fromkeys([*model.obs_keys, 'position', 'direction', 'pixels', 'goal', 'goal_position',
                                   'terminated', 'state']))
        self.cols = {k: dataset.get_col_data(k) for k in keys}
        self.obs = torch.from_numpy(commnet_features(self.cols, model.obs_keys, model.grid_size, goal=False))
        alive = ~np.asarray(self.cols['terminated'], bool)
        rows = []
        for start, length in zip(np.asarray(dataset.offsets), np.asarray(dataset.lengths)):
            last = start + length - 1
            rows += [(t, last) for t in range(start, last) if alive[t].any()]
        self.rows = np.array(rows, np.int64).reshape(-1, 2)

    def __len__(self) -> int:
        return len(self.rows)

    def _goal_rows(self, t: int, last: int, N: int) -> list[int | None]:
        "The row of each agent's goal (None: the episode's goal)."
        if torch.rand(()) >= self.hindsight_prob:
            return [None] * N
        steps = min(self.max_goal_steps, last - t)
        pick = lambda: t + 1 + int(torch.randint(steps, ()))
        if self.hindsight == 'joint':
            return [pick()] * N
        return [pick() for _ in range(N)]

    def __getitem__(self, i: int) -> dict[str, torch.Tensor]:
        t, last = (int(v) for v in self.rows[i])
        c = self.cols
        N = c['position'].shape[1]
        alive = ~np.asarray(c['terminated'][t], bool)
        walkable = walkable_cells(c['state'][t])
        targets, target_dirs, images, labels = [], [], [], np.full(N, IGNORE_INDEX, np.int64)
        for a, g in enumerate(self._goal_rows(t, last, N)):
            if g is None:
                target, target_dir, image = c['goal_position'][t, a], 0, c['goal'][t, a]
            else:
                target, target_dir, image = c['position'][g, a], int(c['direction'][g, a]), c['pixels'][g, a]
            targets.append(np.asarray(target)); target_dirs.append(target_dir); images.append(image)
            if alive[a]:
                actions = shortest_path_actions(walkable, c['position'][t, a], c['direction'][t, a], target)
                if actions is not None and len(actions):
                    labels[a] = actions[0]
        subsets = [self.subsets[int(torch.randint(len(self.subsets), ()))] for _ in range(N)]
        return {
            'obs': self.obs[t],
            'position': torch.from_numpy(np.asarray(c['position'][t], np.int64)),
            'direction': torch.from_numpy(np.asarray(c['direction'][t], np.int64)),
            'alive': torch.from_numpy(alive),
            'goal_pose': torch.from_numpy(goal_pose(c['position'][t], c['direction'][t], np.stack(targets), np.array(target_dirs))),
            'goal_image': torch.from_numpy(np.stack(images)),
            'goal_mask': torch.tensor(subsets),
            'actions': torch.from_numpy(labels),
        }

### Training

`train_gc_commnet` minimizes the cross-entropy of the labels, with the model's communication mask
(alive agents, `comm_range`, `drop_prob`) drawn for every sample. The samples are not in envs, so
comm-core networks are not used in training; a model acts with them (`links`) in a `World`.

In [ ]:
#| export
def train_gc_commnet(
    model: GCCommNet,
    data: GCCommNetDataset,
    steps: int = 5_000, # Gradient steps
    batch_size: int = 64,
    lr: float = 1e-3,
    num_workers: int = 0, # DataLoader workers
    device: str | torch.device | None = None, # None: cuda if available
    log_every: int = 200, # Steps between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the step and the means since the last call
) -> GCCommNet:
    "Train `model` to imitate the labels of `data` (cross-entropy); returns it in eval mode."
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model.to(device).train()
    sampler = torch.utils.data.RandomSampler(data, replacement=True, num_samples=steps * batch_size)
    loader = torch.utils.data.DataLoader(data, batch_size, sampler=sampler, num_workers=num_workers, drop_last=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    totals: dict[str, float] = {}
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        mask = comm_mask(batch['alive'], batch['position'], model.comm_range, model.drop_prob)
        logits = model(batch['obs'], batch['position'], batch['direction'], batch['goal_pose'], batch['goal_image'],
                       batch['goal_mask'], mask)
        labels = batch['actions']
        loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=IGNORE_INDEX)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        valid = labels != IGNORE_INDEX
        row = {'loss': float(loss), 'accuracy': float((logits.argmax(-1) == labels)[valid].float().mean())}
        for k, v in row.items():
            totals[k] = totals.get(k, 0.0) + v
        if log is not None and (step % log_every == 0 or step == steps):
            n = step % log_every or log_every
            log({'step': step, **{k: v / n for k, v in totals.items()}})
            totals = {}
    return model.eval()

### Saving and loading

`GCCommNet` saves and loads as `CommNet` (`save`, `GCCommNet.load`); its checkpoints are marked
`kind='gc'`, which `load_commnet` reads to load either.

In [ ]:
#| export
@patch
def save(self: GCCommNet, path, **extra: Any):
    "Save the config, the weights and `extra` to `path`, marked `kind='gc'`."
    return CommNet.save(self, path, kind='gc', **extra)


def load_commnet(path, map_location: Any = 'cpu') -> CommNet:
    "A `CommNet` or `GCCommNet` saved with `save`."
    kind = torch.load(path, map_location=map_location, weights_only=True).get('kind')
    model = (GCCommNet if kind == 'gc' else CommNet).load(path, map_location)
    model.checkpoint.pop('kind', None)
    return model

### Tests

`MixedPolicy` episodes, hindsight goals and expert labels: a small model with quantized latents,
raw observations and poses learns, acts in a `World` with every goal modality, over comm-core
networks, and loads back.

In [ ]:
#| hide
import tempfile
import comm_core as cc
from stable_marl import World
from stable_marl.data import HDF5Dataset
from stable_marl.envs.multigrid import MixedPolicy
from stable_marl.policy import FeedForwardPolicy
from stable_marl.wm.commnet.core import NetworkLinks

tmp = tempfile.mkdtemp()
kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=15)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
world.set_policy(MixedPolicy(seed=0))
world.collect(f'{tmp}/mixed.h5', episodes=8, seed=0, progress=False)
ds = HDF5Dataset(path=f'{tmp}/mixed.h5')
world.reset(seed=[0, 1])
columns = {k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}
dim = commnet_features(columns, grid_size=7, goal=False).shape[-1]

torch.manual_seed(0)
model = GCCommNet(dim, hidden_dim=32, grid_size=7, share=SHARE, latent_bits=4)
test_eq(model.message_bits(columns), [32 * 4 + 32 * (7 * 7 * 3 + 1) + 32 * 3])   # one round: latent, observation, pose
for hindsight in HINDSIGHT:
    data = GCCommNetDataset(ds, model, hindsight_prob=1.0, hindsight=hindsight)
    s = data[0]
    test_eq(s['obs'].shape, (2, dim)); test_eq(s['goal_image'].shape, (2, 28, 28, 3)); test_eq(s['actions'].shape, (2,))
    assert s['goal_mask'].any(-1).all()
    assert ((s['actions'] == IGNORE_INDEX) | s['alive']).all()       # labels only for agents still acting
data = GCCommNetDataset(ds, model)
logs = []
train_gc_commnet(model, data, steps=80, batch_size=16, lr=3e-3, log_every=40, log=logs.append, device='cpu')
assert logs[-1]['loss'] < logs[0]['loss']

model.temperature = 0.0
world.set_policy(FeedForwardPolicy(model))
for modalities in (('pose',), ('image',), ('pose', 'image')):
    model.eval_modalities = modalities
    test_eq(len(world.evaluate(episodes=2, seed=100)['episode_successes']), 2)
model.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=0.0))
world.evaluate(episodes=2, seed=100)
m = model.links.metrics[0]
assert m.transmissions > 0; test_eq(m.bits_sent, m.transmissions * model.message_bits(columns)[0])

path = model.save(f'{tmp}/commnet.pt', env_name='MultiGrid-FindGoal-15x15-v0')
again = load_commnet(path)
test_eq(type(again), GCCommNet); test_eq(again.share, SHARE); test_eq(again.checkpoint, {'env_name': 'MultiGrid-FindGoal-15x15-v0'})
inputs = {k: v for k, v in model.observe(columns).items()}
test_close(again(**inputs), model.cpu()(**inputs))
test_eq(type(load_commnet(CommNet(dim).save(f'{tmp}/plain.pt'))), CommNet)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()